In [1]:
import pandas as pd
from powerball_ticket_generator import TemperatureLotteryGenerator
from powerball_backtester import PowerballBacktester
from jackpots_scraper import Jackpots

In [2]:
# Update Jackpots dataset
Jackpots().to_csv("jackpots.csv")

'jackpots.csv'

In [3]:
generator = TemperatureLotteryGenerator(
    csv_path="powerball.csv",
    T_white_min=0.0,
    T_red_min=0.0,
    temperature_scale=200.0,          # makes 200 map to alpha≈1
    temperature_sampling="rev_log1p", # mostly high T, few low T
)

tix = generator.generate_ticket_batch(
    n=5,
    max_T=200,
    include_metadata=False
)

tix = pd.DataFrame(tix)
tix

FileNotFoundError: [Errno 2] No such file or directory: 'V2/powerball.csv'


### 1) Baseline single-run backtest (fixed spend, multiplier on, temperatures stored)

In [ ]:
gen = TemperatureLotteryGenerator(
    csv_path="powerball.csv",
    T_white_min=0.0,
    T_red_min=0.0,
    temperature_scale=200.0,          # makes 200 map to alpha≈1
    temperature_sampling="rev_log1p", # mostly high T, few low T
)

bt = PowerballBacktester(
    draw_csv="powerball.csv",
    jackpot_csv="jackpots.csv",
    generator=gen,
    ticket_budget=20,
    use_multiplier=True,
    reinvest_percent=0.5,
    store_temperatures=True,
    rolling_window=100,
    prefer_numba=True,
    withdrawal_apy=0.03,
    max_T=200
)

out = bt.run(seed=None)
bt.plot_winnings(out)

In [ ]:
bt.pnl_table 

### 2) Determinism/regression test harness (repeatability + RNG isolation)

In [ ]:
out1 = bt.run(seed=123456)
ticket1 = bt.last_ticket_detail.copy()

out2 = bt.run(seed=123456)
ticket2 = bt.last_ticket_detail.copy()

assert out1.equals(out2)
assert ticket1.equals(ticket2)
assert out1["net_profit"].iat[-1] == out2["net_profit"].iat[-1]  # scalar compare

### 3) Compare reinvestment policies (fixed_exposure vs nested_compounding)

In [ ]:
# Same ticket count each draw (isolates accounting effects)
df_fixed = bt.compare_reinvest_rates(
    reinvest_rates=(0.0, 0.25, 0.5, 1.0),
    seed=20250101,
    mode="fixed_exposure",
    plot=True,
)

# “Reinvest buys more tickets” while keeping runs comparable via nested pools
df_nested = bt.compare_reinvest_rates(
    reinvest_rates=(0.0, 0.25, 0.5, 1.0),
    seed=20250101,
    mode="nested_compounding",
    plot=True,
)

df_nested.head()

### 4) Model “withdraw winnings into an external account” (withdrawal_apy + draws_per_year)

In [ ]:
bt = PowerballBacktester(
    draw_csv="powerball.csv",
    jackpot_csv="jackpots.csv",
    generator=gen,
    ticket_budget=100,
    reinvest_percent=0.50,   # half reinvest, half withdraw
    withdrawal_apy=0.05,     # external compounding on withdrawals
    draws_per_year=104,      # used to convert APY to per-draw rate
    max_T=200
)

out = bt.run(seed=7)
out

In [ ]:
bt.plot_winnings(out)

### 5) Plot a single run with dual-axis “net profit level vs profit/draw” and cashflows

In [ ]:
out = bt.run(seed=123)
bt.plot_winnings(out)
# optional:
ticket_df = bt.last_ticket_detail
summary = bt.last_summary

### 6) Temperature-stratified performance (deciles of white temperature)

In [ ]:
draw_detail = bt.run(seed=123)  # run() returns draw_detail now :contentReference[oaicite:1]{index=1}
ticket_detail = bt.last_ticket_detail  # populated by run() :contentReference[oaicite:2]{index=2}

summary = bt.summarize_by_white_temperature_deciles(ticket_detail, q=10)
summary

### 7) Performance run: turn off metadata to reduce overhead (store_temperatures=False)

In [ ]:
bt_fast = PowerballBacktester(
    draw_csv="powerball.csv",
    jackpot_csv="jackpots.csv",
    generator=gen,
    ticket_budget=500,
    store_temperatures=False,  # leaner generation; no temp columns
    prefer_numba=True,
)

draw_detail = bt_fast.run(seed=123)  # returns draw_detail :contentReference[oaicite:3]{index=3}
ticket_detail = bt_fast.last_ticket_detail  # populated by run() :contentReference[oaicite:4]{index=4}

print(ticket_detail.columns)  # should not include white_temperature/red_temperature

### 8) Verify multiplier economics and budget allocation behavior

In [ ]:
bt_alloc = PowerballBacktester(
    draw_csv="powerball.csv",
    jackpot_csv="jackpots.csv",
    generator=gen,
    ticket_budget=7,
    use_multiplier=True,
)

# Internal allocation logic: (n_multiplier, n_non_multiplier)
print(bt_alloc._allocate_ticket_counts(7))   # expected pattern like (1, 2)
print(bt_alloc._allocate_ticket_counts(10))  # expected pattern like (2, 2)

### 9) Enforced uniqueness across multiplier and non-multiplier pools (per draw)

In [ ]:
bt = PowerballBacktester(
    draw_csv="powerball.csv",
    jackpot_csv="jackpots.csv",
    generator=gen,            # must support existing_tickets=...
    ticket_budget=101,
    use_multiplier=True,
)

draw_detail = bt.run(seed=99)          # returns draw_detail :contentReference[oaicite:5]{index=5}
td = bt.last_ticket_detail             # ticket-level rows :contentReference[oaicite:6]{index=6}

dupes = td.duplicated(
    subset=["date", "white_1", "white_2", "white_3", "white_4", "white_5", "red_ball"]
).sum()

print("duplicate tickets within a draw:", dupes)

### 10) Use the generator standalone (CSV-friendly ticket export + uniqueness controls)

In [ ]:
from V2.powerball_ticket_generator import TemperatureLotteryGenerator

gen = TemperatureLotteryGenerator(
    csv_path="powerball.csv",
    T_white_min=0.0,
    T_red_min=0.0,
    temperature_scale=200.0,          # makes 200 map to alpha≈1
    temperature_sampling="rev_log1p", # mostly high T, few low T
)

# Cashier-friendly output (flat columns)
tickets = gen.generate_ticket_batch(
    n=10,
    max_T=50.0,
    include_metadata=False,
    ensure_unique=True,
)

# Export-ready
df = pd.DataFrame(tickets)
df.to_csv("tickets.csv", index=False)

# Enforce uniqueness vs an existing pool (e.g., “don’t repeat last week’s tickets”)
more = gen.generate_ticket_batch(
    n=10,
    max_T=50.0,
    include_metadata=True,
    ensure_unique=True,
    existing_tickets=tickets,
)

df

In [ ]:
pd.DataFrame(more)